**Task 1: Model Training**

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv('/content/train.csv')

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Select input features
features = ['Pclass', 'Sex', 'Age', 'SibSp',
            'Parch', 'Fare', 'Embarked']

X = df[features]

# Target variable
y = df['Survived']

# Numerical and categorical columns
num_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
cat_cols = ['Sex', 'Embarked']

# Preprocessing for numerical data
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Preprocessing for categorical data
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

Training Records: 712
Testing Records: 179


In [5]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train the model
model.fit(X_train, y_train)

print("Model Training Completed!")

Model Training Completed!


In [6]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.8156424581005587
Accuracy Percentage: 81.56424581005587

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179



In [7]:
joblib.dump(model, 'model.pkl')

print("Model saved successfully as model.pkl")

Model saved successfully as model.pkl


**Task 2: Online Inference**

In [8]:
# Load saved model
loaded_model = joblib.load('model.pkl')

# Select one unseen passenger
online_record = X_test.iloc[[0]]

# Generate prediction
prediction = loaded_model.predict(online_record)

print("Passenger Details:")
print(online_record)

if prediction[0] == 1:
    print("\nPrediction: Survived")
else:
    print("\nPrediction: Not Survived")

Passenger Details:
     Pclass   Sex   Age  SibSp  Parch   Fare Embarked
565       3  male  24.0      2      0  24.15        S

Prediction: Not Survived


In [9]:
probability = loaded_model.predict_proba(online_record)

print("Survival Probability:", probability[0][1])
print("Non-Survival Probability:", probability[0][0])

Survival Probability: 0.17
Non-Survival Probability: 0.83


**Task 3: Batch Inference**

In [10]:
# Select 20 unseen records
batch_records = X_test.iloc[:20]

# Predict all 20 records together
batch_predictions = loaded_model.predict(batch_records)

# Create results dataframe
batch_results = batch_records.copy()

batch_results['Actual_Survived'] = y_test.iloc[:20].values
batch_results['Predicted_Survived'] = batch_predictions

# Display results
print(batch_results)

     Pclass     Sex   Age  SibSp  Parch      Fare Embarked  Actual_Survived  \
565       3    male  24.0      2      0   24.1500        S                0   
160       3    male  44.0      0      1   16.1000        S                0   
553       3    male  22.0      0      0    7.2250        C                1   
860       3    male  41.0      2      0   14.1083        S                0   
241       3  female   NaN      1      0   15.5000        Q                1   
559       3  female  36.0      1      0   17.4000        S                1   
387       2  female  36.0      0      0   13.0000        S                1   
536       1    male  45.0      0      0   26.5500        S                0   
698       1    male  49.0      1      1  110.8833        C                0   
99        2    male  34.0      1      0   26.0000        S                0   
126       3    male   NaN      0      0    7.7500        Q                0   
333       3    male  16.0      2      0   18.0000   

In [11]:
batch_results.to_csv(
    'batch_predictions.csv',
    index=False
)

print("Batch predictions saved successfully!")

Batch predictions saved successfully!


In [12]:
from google.colab import files

files.download('batch_predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Task 4: Streaming Inference**

In [13]:
import time

# Select 10 unseen records
stream_records = X_test.iloc[20:30]

print("Streaming Inference Started\n")

for i in range(len(stream_records)):

    # Get one record
    record = stream_records.iloc[[i]]

    # Predict
    prediction = loaded_model.predict(record)

    # Display result
    print("Incoming Passenger:", i + 1)

    if prediction[0] == 1:
        print("Prediction: Survived")
    else:
        print("Prediction: Not Survived")

    print("-" * 30)

    # Delay of 1 second
    time.sleep(1)

print("Streaming Inference Completed!")

Streaming Inference Started

Incoming Passenger: 1
Prediction: Not Survived
------------------------------
Incoming Passenger: 2
Prediction: Survived
------------------------------
Incoming Passenger: 3
Prediction: Not Survived
------------------------------
Incoming Passenger: 4
Prediction: Survived
------------------------------
Incoming Passenger: 5
Prediction: Not Survived
------------------------------
Incoming Passenger: 6
Prediction: Survived
------------------------------
Incoming Passenger: 7
Prediction: Not Survived
------------------------------
Incoming Passenger: 8
Prediction: Not Survived
------------------------------
Incoming Passenger: 9
Prediction: Not Survived
------------------------------
Incoming Passenger: 10
Prediction: Not Survived
------------------------------
Streaming Inference Completed!


**Task 5: Model Serving using FastAPI**

In [14]:
!pip install fastapi uvicorn requests

In [15]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import pandas as pd

app = FastAPI(
    title="Titanic Survival Prediction API",
    description="ML Model Serving using FastAPI",
    version="1.0"
)

# Load trained model
model = joblib.load("model.pkl")


# Input data validation
class Passenger(BaseModel):

    Pclass: int = Field(ge=1, le=3)
    Sex: str
    Age: float = Field(ge=0, le=100)
    SibSp: int = Field(ge=0)
    Parch: int = Field(ge=0)
    Fare: float = Field(ge=0)
    Embarked: str


# Health endpoint
@app.get("/health")
def health():

    return {
        "status": "API is running"
    }


# Prediction endpoint
@app.post("/predict")
def predict(passenger: Passenger):

    # Convert input into dataframe
    data = pd.DataFrame([passenger.model_dump()])

    # Generate prediction
    prediction = model.predict(data)

    # Return result
    if prediction[0] == 1:
        result = "Survived"
    else:
        result = "Not Survived"

    return {
        "prediction": result,
        "survived": int(prediction[0])
    }

Writing app.py


In [16]:
%%writefile client.py

import requests

url = "http://127.0.0.1:8000/predict"

passenger_data = {
    "Pclass": 3,
    "Sex": "male",
    "Age": 25,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S"
}

response = requests.post(
    url,
    json=passenger_data
)

print("Status Code:", response.status_code)

print("Prediction:")
print(response.json())

Writing client.py


In [17]:
%%writefile requirements.txt

pandas
numpy
scikit-learn
joblib
fastapi
uvicorn
requests

Writing requirements.txt


In [1]:
!uvicorn app:app --host 0.0.0.0 --port 8000

INFO:     Started server process [8359]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [8359]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "uvloop/loop.pyx", line 1509, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1502, in uvloop.loop.Loop.run_until_complete
    self.run_forever()
  File "uvloop/loop.pyx", line 1371, in uvloop.loop.Loop.run_forever
    self._run(mode)
  File "uvloop/loop.pyx", line 551, in uvloop.loop.Loop._run
    raise self._last_error
  File "uvloop/loop.pyx", line 471, in uvloop.loop.Loop._on_i

In [19]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8000)")

print(url)

https://8000-m-s-use1b1-30nofhdk3x77n-b.us-east1.prod.r.colab.dev


In [1]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > /tmp/uvicorn.log 2>&1 &

In [2]:
import requests

response = requests.get("http://127.0.0.1:8000/health")

print(response.status_code)
print(response.json())

200
{'status': 'API is running'}


In [3]:
import requests

url = "http://127.0.0.1:8000/predict"

passenger_data = {
    "Pclass": 3,
    "Sex": "male",
    "Age": 25,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S"
}

response = requests.post(
    url,
    json=passenger_data
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

Status Code: 200
Prediction: {'prediction': 'Not Survived', 'survived': 0}


In [4]:
invalid_data = {
    "Pclass": 3,
    "Sex": "male",
    "Age": -5,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S"
}

response = requests.post(
    url,
    json=invalid_data
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 422
Response: {'detail': [{'type': 'greater_than_equal', 'loc': ['body', 'Age'], 'msg': 'Input should be greater than or equal to 0', 'input': -5, 'ctx': {'ge': 0.0}}]}


In [5]:
invalid_data = {
    "Pclass": 3,
    "Sex": "male",
    "Age": -5,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S"
}

response = requests.post(
    url,
    json=invalid_data
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 422
Response: {'detail': [{'type': 'greater_than_equal', 'loc': ['body', 'Age'], 'msg': 'Input should be greater than or equal to 0', 'input': -5, 'ctx': {'ge': 0.0}}]}


In [6]:
from google.colab import files

files.download('model.pkl')
files.download('batch_predictions.csv')
files.download('app.py')
files.download('client.py')
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Task 6: Observations**

1. How does Online Inference differ from Batch Inference?

Online inference predicts one record at a time, whereas batch inference predicts multiple records together.

2. What is the difference between Streaming Inference and Batch Inference?

Streaming inference processes incoming records continuously, one by one. Batch inference processes a group of records together.

3. Why is FastAPI used for model serving?

FastAPI is used to expose the trained ML model through REST API endpoints. It supports data validation and provides fast API communication.

4. Why should the same trained model be reused for all inference strategies?

The same trained model ensures consistent predictions and avoids differences caused by retraining.